# Return Expectations — Model Selection Notebook

Runs the same model comparison used for earnings/dividend expectations to
determine which model best fits each return expectation series.

**Two target series:**
- `sp_1yr_median`  — median 1-year S&P 500 return expectation (%)
- `sp_10yr_median` — median 10-year S&P 500 return expectation (%)

**Models:** Ridge, Random Forest, PLS, PLS->Ridge, PLS->KernelRidge

**Requires:** `news_df`, `article_embeddings`, `p_neutral_all`, `shiller_df`,
`aggregate_to_waves`, `get_shiller_features_for_wave` from main pipeline.
Copy the returns Excel file to `data/return_expectations_quarterly.xlsx`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, pickle
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, KFold
from sklearn.metrics import mean_squared_error

SEED = 42
np.random.seed(SEED)

DATA_DIR   = Path('./data')
OUTPUT_DIR = Path('./output/returns')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

RETURNS_EXCEL  = DATA_DIR / 'return_expectations_quarterly.xlsx'
TARGET_SERIES  = ['sp_1yr_mean', 'sp_10yr_mean']

AGG_WINDOW_DAYS       = 30
RECENCY_WEIGHTING     = 'exponential'
RECENCY_HALFLIFE_DAYS = 7
MIN_TRAIN_WAVES       = 40
NEUTRAL_THRESHOLD     = 0.90

RIDGE_ALPHAS         = np.logspace(-3, 6, 50)
RF_N_ESTIMATORS      = 400
RF_MSL_GRID          = [2, 3, 5, 8, 13, 20, 30]
RF_MAX_FEATURES_GRID = ['sqrt', 'log2', 0.05, 0.10]
PLS_MAX_K            = 8

try:
    N_MACRO = len(SHILLER_FEATURES)
except NameError:
    SHILLER_FEATURES = [
        'dp_ratio', 'ep_ratio', 'cape_inv', 'gs10',
        'infl_yoy', 'real_price_gr', 'div_gr_yoy', 'earn_gr_yoy',
    ]
    N_MACRO = len(SHILLER_FEATURES)

print(f'Config loaded.  N_MACRO={N_MACRO}')

## 2. Load return expectations data

In [ ]:
def load_return_expectations(path=RETURNS_EXCEL):
    df = pd.read_excel(path)
    df.columns = df.columns.str.strip()
    df['wave_date'] = pd.to_datetime(
        df.apply(lambda r: f"{int(r['year'])}-{int(r['qtr'])*3:02d}-01", axis=1)
    ) + pd.offsets.MonthEnd(0)
    df = df.sort_values('wave_date').reset_index(drop=True)

    # Fix anomalous 2013Q4 sp_10yr_mean value (outlier near 0)
    # Interpolate linearly from surrounding quarters
    outlier_mask = df['sp_10yr_mean'] < 1.0
    if outlier_mask.any():
        print(f'Interpolating {outlier_mask.sum()} outlier(s) in sp_10yr_mean:')
        for idx in df[outlier_mask].index:
            prev_val = df.loc[idx-1, 'sp_10yr_mean'] if idx > 0 else np.nan
            next_val = df.loc[idx+1, 'sp_10yr_mean'] if idx < len(df)-1 else np.nan
            interp_val = np.nanmean([prev_val, next_val])
            print(f'  {df.loc[idx, "wave_date"].date()}: '
                  f'{df.loc[idx, "sp_10yr_mean"]:.4f} -> {interp_val:.4f} '
                  f'(avg of {prev_val:.4f}, {next_val:.4f})')
            df.loc[idx, 'sp_10yr_mean'] = interp_val

    # Flag interpolated rows from source data
    df['source_interpolated'] = df['interpolated'].astype(bool)

    print(f'Loaded {len(df)} quarters  '
          f'({df["wave_date"].min().date()} to {df["wave_date"].max().date()})')
    print(f'  Source-interpolated rows: {df["source_interpolated"].sum()}')
    print(f'  1yr mean range:  {df["sp_1yr_mean"].min():.2f} - {df["sp_1yr_mean"].max():.2f}%')
    print(f'  10yr mean range: {df["sp_10yr_mean"].min():.2f} - {df["sp_10yr_mean"].max():.2f}%')
    return df

returns_df = load_return_expectations()

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
fig.suptitle('S&P 500 Return Expectations (Mean)', fontweight='bold')
for ax, col in zip(axes, ['sp_1yr_mean', 'sp_10yr_mean']):
    ax.plot(returns_df['wave_date'], returns_df[col], 'o-', ms=3, lw=1.5, color='#2c5f8a')
    interp = returns_df[returns_df['source_interpolated']]
    if len(interp): ax.scatter(interp['wave_date'], interp[col], color='orange',
                               s=40, zorder=5, label='source-interpolated')
    ax.set_ylabel(col); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'return_expectations_raw.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Aggregate embeddings for both series

In [ ]:
def build_survey_df(returns_df, target_col,
                    exclude_interpolated=True):
    df = returns_df.copy()
    if exclude_interpolated:
        df = df[~df['source_interpolated']]
    df = df[['wave_date', target_col]].dropna()
    return df.rename(columns={target_col: 'median_forecast'}).reset_index(drop=True)

wave_data = {}
for target_col in TARGET_SERIES:
    survey_df = build_survey_df(returns_df, target_col)
    print(f'{target_col}: {len(survey_df)} clean waves')
    Z, wdf = aggregate_to_waves(
        news_df, article_embeddings, survey_df,
        p_neutral_all=p_neutral_all,
        neutral_threshold=NEUTRAL_THRESHOLD,
        shiller_df=shiller_df,
    )
    col_means = np.nanmean(Z[:MIN_TRAIN_WAVES, -N_MACRO:], axis=0)
    for j in range(N_MACRO):
        mask = np.isnan(Z[:, -N_MACRO + j])
        Z[mask, -N_MACRO + j] = col_means[j]
    wave_data[target_col] = {'Z': Z, 'y': wdf['median_forecast'].values, 'waves_df': wdf}
    print(f'  Z: {Z.shape}  y: {wdf["median_forecast"].min():.2f} - '
          f'{wdf["median_forecast"].max():.2f}%  '
          f'articles/wave: {wdf["n_after_filter"].median():.0f}')

## 4. Walk-forward harness and model definitions

In [ ]:
def walk_forward(Z, y, min_train, model_fn, scale=True):
    records = []
    for t in range(min_train, len(y)):
        Z_tr, y_tr = Z[:t], y[:t]
        Z_va = Z[t:t+1]
        if scale:
            sc = StandardScaler().fit(Z_tr)
            Z_tr = sc.transform(Z_tr)
            Z_va = sc.transform(Z_va)
        m = model_fn(Z_tr, y_tr)
        records.append({'realized': float(y[t]),
                         'predicted': float(m.predict(Z_va).ravel()[0])})
    wf = pd.DataFrame(records)
    wf['error']        = wf['realized'] - wf['predicted']
    wf['sq_error']     = wf['error'] ** 2
    wf['ar1_pred']     = np.concatenate([[y[min_train-1]], wf['realized'].values[:-1]])
    wf['ar1_sq_error'] = (wf['realized'] - wf['ar1_pred']) ** 2
    rmse     = np.sqrt(wf['sq_error'].mean())
    ar1_rmse = np.sqrt(wf['ar1_sq_error'].mean())
    y_wf = wf['realized'].values
    r2   = 1 - wf['sq_error'].sum() / np.sum((y_wf - y_wf.mean())**2)
    dy_t = np.diff(y_wf); dy_p = np.diff(wf['predicted'].values)
    dir_acc = np.mean(np.sign(dy_t) == np.sign(dy_p)) if len(dy_t) > 0 else np.nan
    return {'rmse': rmse, 'ar1_rmse': ar1_rmse, 'r2': r2,
            'dir_acc': dir_acc, 'wf_df': wf}


def tune_once(Z_train, y_train):
    n  = len(y_train)
    cv = LeaveOneOut() if n < 10 else KFold(n_splits=5, shuffle=False)
    # Ridge
    sc = StandardScaler().fit(Z_train)
    rc = RidgeCV(alphas=RIDGE_ALPHAS, gcv_mode='auto',
                 scoring='neg_mean_squared_error')
    rc.fit(sc.transform(Z_train), y_train)
    best_alpha = rc.alpha_
    # RF
    best_rf_mse, best_msl, best_mf = np.inf, RF_MSL_GRID[0], RF_MAX_FEATURES_GRID[0]
    for msl in RF_MSL_GRID:
        for mf in RF_MAX_FEATURES_GRID:
            preds = np.empty(n)
            for tr, va in cv.split(Z_train):
                m = RandomForestRegressor(n_estimators=100, min_samples_leaf=msl,
                    max_features=mf, random_state=SEED, n_jobs=-1)
                m.fit(Z_train[tr], y_train[tr])
                preds[va] = m.predict(Z_train[va])
            mse = mean_squared_error(y_train, preds)
            if mse < best_rf_mse: best_rf_mse, best_msl, best_mf = mse, msl, mf
    # PLS
    max_k = min(PLS_MAX_K, n - 2, Z_train.shape[1])
    best_pls_mse, best_k = np.inf, 1
    for k in range(1, max_k + 1):
        preds = np.empty(n)
        for tr, va in LeaveOneOut().split(Z_train):
            m = PLSRegression(n_components=k, scale=True)
            m.fit(Z_train[tr], y_train[tr])
            preds[va] = m.predict(Z_train[va]).ravel()
        mse = mean_squared_error(y_train, preds)
        if mse < best_pls_mse: best_pls_mse, best_k = mse, k
    print(f'  Tuned: Ridge a={best_alpha:.2e}  RF msl={best_msl} mf={best_mf}  PLS k={best_k}')
    return best_alpha, best_msl, best_mf, best_k


def make_models(alpha, msl, mf, k):
    def model_ridge(Z_tr, y_tr):
        return Ridge(alpha=alpha).fit(Z_tr, y_tr)
    def model_rf(Z_tr, y_tr):
        return RandomForestRegressor(n_estimators=RF_N_ESTIMATORS,
            min_samples_leaf=msl, max_features=mf,
            random_state=SEED, n_jobs=-1).fit(Z_tr, y_tr)
    def model_pls(Z_tr, y_tr):
        return PLSRegression(n_components=k, scale=True).fit(Z_tr, y_tr)

    class PLSRidgeStack:
        def fit(self, Z_tr, y_tr):
            self.pls = PLSRegression(n_components=k, scale=True).fit(Z_tr[:, :-N_MACRO], y_tr)
            Zp = np.hstack([self.pls.transform(Z_tr[:, :-N_MACRO]), Z_tr[:, -N_MACRO:]])
            self.sc = StandardScaler().fit(Zp)
            self.ridge = Ridge(alpha=alpha).fit(self.sc.transform(Zp), y_tr)
            return self
        def predict(self, Z_va):
            Zp = np.hstack([self.pls.transform(Z_va[:, :-N_MACRO]), Z_va[:, -N_MACRO:]])
            return self.ridge.predict(self.sc.transform(Zp))
    def model_pls_ridge(Z_tr, y_tr): return PLSRidgeStack().fit(Z_tr, y_tr)

    class PLSKernelRidgeStack:
        def fit(self, Z_tr, y_tr):
            self.pls = PLSRegression(n_components=k, scale=True).fit(Z_tr[:, :-N_MACRO], y_tr)
            Zp = np.hstack([self.pls.transform(Z_tr[:, :-N_MACRO]), Z_tr[:, -N_MACRO:]])
            self.sc = StandardScaler().fit(Zp)
            self.kr = KernelRidge(kernel='rbf', alpha=0.1).fit(self.sc.transform(Zp), y_tr)
            return self
        def predict(self, Z_va):
            Zp = np.hstack([self.pls.transform(Z_va[:, :-N_MACRO]), Z_va[:, -N_MACRO:]])
            return self.kr.predict(self.sc.transform(Zp))
    def model_pls_kr(Z_tr, y_tr): return PLSKernelRidgeStack().fit(Z_tr, y_tr)

    return {
        'Ridge':              (model_ridge,     True),
        'Random Forest':      (model_rf,        False),
        'PLS':                (model_pls,       False),
        'PLS->Ridge':         (model_pls_ridge, False),
        'PLS->KernelRidge':   (model_pls_kr,    False),
    }

print('Harness ready.')

## 5. Run all models

In [ ]:
all_wf_results = {}

for target_col in TARGET_SERIES:
    print(f'\n{"="*60}')
    print(f'  {target_col}')
    print(f'{"="*60}')
    Z = wave_data[target_col]['Z']
    y = wave_data[target_col]['y']
    n = len(y)
    print(f'  n={n}  eval steps={n - MIN_TRAIN_WAVES}')
    print(f'  Tuning on first {MIN_TRAIN_WAVES} waves...')
    alpha, msl, mf, k = tune_once(Z[:MIN_TRAIN_WAVES], y[:MIN_TRAIN_WAVES])
    models = make_models(alpha, msl, mf, k)
    series_results = {}
    for name, (fn, do_scale) in models.items():
        print(f'  Running {name}...', end=' ', flush=True)
        res = walk_forward(Z, y, MIN_TRAIN_WAVES, fn, scale=do_scale)
        series_results[name] = res
        print(f'R2={res["r2"]:.3f}  RMSE={res["rmse"]:.4f}  DirAcc={res["dir_acc"]:.1%}')
    all_wf_results[target_col] = {
        'results': series_results, 'Z': Z, 'y': y,
        'waves_df': wave_data[target_col]['waves_df'],
        'best_params': {'ridge_alpha': alpha, 'rf_msl': msl, 'rf_mf': mf, 'pls_k': k},
    }

In [ ]:
model_colors = {
    'Ridge': '#2c5f8a', 'Random Forest': '#c0392b',
    'PLS': '#4a9e6b', 'PLS->Ridge': '#e07b39', 'PLS->KernelRidge': '#9b59b6',
}

for target_col in TARGET_SERIES:
    data = all_wf_results[target_col]
    results = data['results']
    y = data['y']; waves_df = data['waves_df']
    wf_dates = waves_df['wave_date'].values[MIN_TRAIN_WAVES:]
    y_eval = y[MIN_TRAIN_WAVES:]
    ar1_rmse = list(results.values())[0]['ar1_rmse']

    # Summary table
    print(f'\n-- {target_col} -----------------------------------------')
    print(f'{"Model":<22} {"R2":>8} {"RMSE":>8} {"vs AR(1)":>10} {"DirAcc":>9}')
    print('-' * 62)
    print(f'{"AR(1)":<22} {"0.000":>8} {ar1_rmse:>8.4f} {"--":>10} {"--":>9}')
    for name, res in results.items():
        delta = res['rmse'] - ar1_rmse
        sign  = '+' if delta >= 0 else ''
        print(f'{name:<22} {res["r2"]:>8.3f} {res["rmse"]:>8.4f} '
              f'{sign+f"{delta:.4f}":>10} {res["dir_acc"]:>8.1%}')

    # Plots
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle(f'{target_col}  (n_eval={len(y_eval)}, min_train={MIN_TRAIN_WAVES})',
                 fontweight='bold')
    ax = axes[0]
    names  = ['AR(1)'] + list(results.keys())
    rmses  = [ar1_rmse] + [results[m]['rmse'] for m in results]
    colors = ['#7f8c8d'] + [model_colors[m] for m in results]
    bars = ax.bar(names, rmses, color=colors, alpha=0.88, edgecolor='white', width=0.6)
    for bar, val in zip(bars, rmses):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)
    ax.axhline(ar1_rmse, color='#7f8c8d', ls='--', lw=1, alpha=0.6)
    ax.set_title('RMSE'); ax.set_xticklabels(names, rotation=30, ha='right', fontsize=7)
    ax.grid(alpha=0.25, axis='y')
    ax = axes[1]
    r2s  = [0.0] + [results[m]['r2'] for m in results]
    bars = ax.bar(names, r2s, color=colors, alpha=0.88, edgecolor='white', width=0.6)
    ax.axhline(0, color='black', lw=0.8, ls='--')
    for bar, val in zip(bars, r2s):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)
    ax.set_title('R2  (AR(1) = 0)'); ax.set_xticklabels(names, rotation=30, ha='right', fontsize=7)
    ax.grid(alpha=0.25, axis='y')
    ax = axes[2]
    ax.plot(wf_dates, y_eval, '-', color='black', lw=2, label='Realized', zorder=6)
    for name, res in results.items():
        ax.plot(wf_dates, res['wf_df']['predicted'].values, '--',
                color=model_colors[name], alpha=0.75, lw=1.2, label=name)
    ax.set_title('Walk-forward predictions'); ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'model_comparison_{target_col}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
ROLL = 10

for target_col in TARGET_SERIES:
    data = all_wf_results[target_col]
    results = data['results']
    y = data['y']; waves_df = data['waves_df']
    wf_dates = waves_df['wave_date'].values[MIN_TRAIN_WAVES:]
    y_eval = y[MIN_TRAIN_WAVES:]
    y_prev = np.concatenate([[y[MIN_TRAIN_WAVES-1]], y_eval[:-1]])
    fig, ax = plt.subplots(figsize=(14, 5))
    fig.suptitle(f'Rolling {ROLL}-step RMSE -- {target_col}', fontweight='bold')
    for name, res in results.items():
        sq_err  = (y_eval - res['wf_df']['predicted'].values) ** 2
        rolling = pd.Series(sq_err).rolling(ROLL, min_periods=3).mean().pipe(np.sqrt)
        ax.plot(wf_dates, rolling, lw=1.8, color=model_colors[name], label=name)
    ar1_sq   = (y_eval - y_prev) ** 2
    ar1_roll = pd.Series(ar1_sq).rolling(ROLL, min_periods=3).mean().pipe(np.sqrt)
    ax.plot(wf_dates, ar1_roll, '-', color='gray', lw=1.5, alpha=0.7, label='AR(1)')
    ax.set_xlabel('Wave date'); ax.set_ylabel(f'Rolling {ROLL}-step RMSE')
    ax.legend(fontsize=8); ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'rolling_rmse_{target_col}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
print('\n-- Model Recommendation ------------------------------------------')
for target_col in TARGET_SERIES:
    results  = all_wf_results[target_col]['results']
    bp       = all_wf_results[target_col]['best_params']
    ar1_rmse = list(results.values())[0]['ar1_rmse']
    best_r2   = max(results.items(), key=lambda x: x[1]['r2'])
    best_rmse = min(results.items(), key=lambda x: x[1]['rmse'])
    best_dir  = max(results.items(), key=lambda x: x[1]['dir_acc'])
    print(f'\n  {target_col}  AR(1) RMSE={ar1_rmse:.4f}')
    print(f'  Best R2:        {best_r2[0]:<22} R2={best_r2[1]["r2"]:.3f}')
    print(f'  Best RMSE:      {best_rmse[0]:<22} RMSE={best_rmse[1]["rmse"]:.4f}')
    print(f'  Best dir acc:   {best_dir[0]:<22} {best_dir[1]["dir_acc"]:.1%}')
    print(f'  Params: {bp}')
    beats_ar1 = [n for n, r in results.items() if r['rmse'] < ar1_rmse]
    if beats_ar1: print(f'  Models beating AR(1): {beats_ar1}')
    else: print(f'  Warning: no model beats AR(1) on RMSE')

print('\nTo integrate into main pipeline add to:')
print('  SURVEY_SERIES  -- sp_1yr_median and/or sp_10yr_median loaders')
print('  MODEL_CHOICE   -- best model per series')
print('  MIN_TRAIN_WAVES_PER_SERIES')